In [1]:
import os

In [2]:
os.chdir("../")
%pwd

'd:\\AI Projects\\Thesis project'

In [7]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    scaler_X_path: Path
    scaler_y_path: Path
    metric_file_name: Path
    mlflow_uri: str
    
    # We need the architecture params to rebuild the model before loading the weights
    input_dim: int
    output_dim: int
    hidden_layers: list
    seq_length: int
    target_feature: str
    collocation_flag: str
    input_features: list

In [8]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories
# from mlProject.entity.config_entity import ModelTrainerConfig # Make sure to import the new dataclass
from pathlib import Path

class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([Path(self.config.artifacts_root)])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.model_training
        schema = self.schema

        create_directories([config.root_dir])

        target_col = schema.target_column.name
        all_columns = list(schema.columns.keys())
        input_feats = [col for col in all_columns if col not in [target_col, "station_name"]]

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=Path(config.root_dir),
            test_data_path=Path(config.test_data_path),
            model_path=Path(config.model_path),
            scaler_X_path=Path(config.scaler_X_path),
            scaler_y_path=Path(config.scaler_y_path),
            metric_file_name=Path(config.metric_file_name),
            mlflow_uri=config.mlflow_uri,
            
            input_dim=params.input_dim,
            output_dim=params.output_dim,
            hidden_layers=list(params.hidden_layers),
            seq_length=params.seq_length,
            target_feature=target_col,
            collocation_flag="t_surf_is_collocation",
            input_features=input_feats
        )

        return model_evaluation_config

In [ ]:
import pandas as pd
import numpy as np
import torch
import joblib
import mlflow
import mlflow.pytorch
from urllib.parse import urlparse
from sklearn.metrics import mean_squared_error, mean_absolute_error
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from mlProject import logger
from mlProject.utils.common import save_json
from mlProject.components.model_training import AdvancedPINN

class ModelEvaluation:
    def __init__(self, config):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def create_sequences(self, data, targets, masks, stations, dates, seq_length):
        """Upgraded to track the Station Name and Date for each sequence"""
        xs, ys, ms, st, dt = [], [], [], [], []
        for i in range(len(data) - seq_length):
            xs.append(data[i:(i + seq_length)])
            ys.append(targets[i + seq_length - 1]) 
            ms.append(masks[i + seq_length - 1])
            st.append(stations[i + seq_length - 1])
            dt.append(dates[i + seq_length - 1])
        return np.array(xs), np.array(ys), np.array(ms), np.array(st), np.array(dt)

    def evaluate_metrics(self, actual, predicted):
        rmse = np.sqrt(mean_squared_error(actual, predicted))
        mae = mean_absolute_error(actual, predicted)
        pvr = (np.sum(predicted > 0.05) / len(predicted)) * 100 # Physical Violation Rate
        return rmse, mae, pvr

    def generate_thesis_plots(self, df_results, scores):
        """Generates 3 scientific plots for the thesis and saves them."""
        logger.info("Generating thesis-ready graphs...")
        sns.set_theme(style="whitegrid")
        
        # Plot 1: Spatial Performance (RMSE by Station)
        plt.figure(figsize=(8, 5))
        stations = ['KAN_L', 'KAN_M', 'EGP']
        rmses = [scores.get(f"RMSE_{s}", 0) for s in stations]
        sns.barplot(x=stations, y=rmses, palette="Blues_d")
        plt.title("Model Accuracy across Elevation Gradients (Spatial Generalization)")
        plt.ylabel("Root Mean Square Error (°C)")
        plt.xlabel("PROMICE Station")
        plt.savefig(Path(self.config.root_dir) / "spatial_rmse_plot.png", dpi=300, bbox_inches='tight')
        plt.close()

        # Plot 2: Actual vs Predicted Scatter (Colored by Season)
        plt.figure(figsize=(8, 8))
        sns.scatterplot(data=df_results, x='Actual', y='Predicted', hue='Season', alpha=0.5, palette={"Summer": "red", "Winter": "blue"})
        plt.plot([-40, 5], [-40, 5], color='black', linestyle='--') # Perfect prediction line
        plt.axhline(0, color='red', linestyle=':', label="0°C Physical Boundary")
        plt.title("LSTM-PINN Prediction Accuracy: Actual vs. Predicted Temperature")
        plt.xlabel("Actual Surface Temperature (°C)")
        plt.ylabel("Predicted Surface Temperature (°C)")
        plt.legend()
        plt.savefig(Path(self.config.root_dir) / "actual_vs_predicted.png", dpi=300, bbox_inches='tight')
        plt.close()

        # Plot 3: Training Loss Curve (If available)
        loss_path = Path("artifacts/model_trainer/training_loss_history.csv")
        if loss_path.exists():
            loss_df = pd.read_csv(loss_path)
            plt.figure(figsize=(10, 5))
            sns.lineplot(data=loss_df, x='epoch', y='loss', color='darkgreen')
            plt.title("Advanced PINN Training Convergence (Total Loss)")
            plt.ylabel("Loss")
            plt.xlabel("Epochs")
            plt.savefig(Path(self.config.root_dir) / "training_loss_curve.png", dpi=300, bbox_inches='tight')
            plt.close()

    def log_into_mlflow(self):
        logger.info("Starting Advanced Evaluation and MLflow logging...")
        
        # 1. Load Data & Scalers
        df = pd.read_csv(self.config.test_data_path)
        df['time'] = pd.to_datetime(df['time']) # Ensure time is datetime
        
        df[self.config.collocation_flag] = df[self.config.target_feature].isna()
        df[self.config.target_feature] = df[self.config.target_feature].fillna(0.0)

        X_raw = df[self.config.input_features].values
        y_raw = df[self.config.target_feature].values.reshape(-1, 1)
        mask_raw = (~df[self.config.collocation_flag]).values.reshape(-1, 1)
        
        # Extract metadata for evaluation
        station_raw = df['station_name'].values
        dates_raw = df['time'].values

        scaler_X = joblib.load(self.config.scaler_X_path)
        scaler_y = joblib.load(self.config.scaler_y_path)

        X_scaled = scaler_X.transform(X_raw)
        y_scaled = scaler_y.transform(y_raw)

        # Create Sequences with Metadata
        seq_X, seq_y, seq_mask, seq_st, seq_dt = self.create_sequences(
            X_scaled, y_scaled, mask_raw, station_raw, dates_raw, self.config.seq_length
        )

        X_tensor = torch.tensor(seq_X, dtype=torch.float32).to(self.device)

        # 2. Rebuild Model and Make Predictions
        model = AdvancedPINN(
            input_dim=self.config.input_dim, 
            hidden_dim=self.config.hidden_layers[0], 
            output_dim=self.config.output_dim
        ).to(self.device)
        model.load_state_dict(torch.load(self.config.model_path, map_location=self.device))
        model.eval()

        with torch.no_grad():
            preds = model(X_tensor)
            pred_t_surf_scaled = preds[:, 0].cpu().numpy().reshape(-1, 1)

        # 3. Unscale and create Results DataFrame
        pred_t_surf = scaler_y.inverse_transform(pred_t_surf_scaled).flatten()
        actual_t_surf = scaler_y.inverse_transform(seq_y).flatten()
        valid_mask = seq_mask.flatten()

        # Build an evaluation dataframe only for valid sensor days
        results_df = pd.DataFrame({
            'Actual': actual_t_surf[valid_mask],
            'Predicted': pred_t_surf[valid_mask],
            'Station': seq_st[valid_mask],
            'Date': pd.to_datetime(seq_dt[valid_mask])
        })
        
        # Define Summer (May to Sept) vs Winter
        results_df['Month'] = results_df['Date'].dt.month
        results_df['Season'] = np.where(results_df['Month'].isin([5, 6, 7, 8, 9]), 'Summer', 'Winter')

        # 4. Calculate Master Metrics
        overall_rmse, overall_mae, overall_pvr = self.evaluate_metrics(results_df['Actual'], results_df['Predicted'])
        scores = {"Overall_RMSE": overall_rmse, "Overall_MAE": overall_mae, "Overall_PVR": overall_pvr}

        # Calculate Spatial (Station) Metrics
        for station in ['KAN_L', 'KAN_M', 'EGP']:
            st_data = results_df[results_df['Station'] == station]
            if not st_data.empty:
                r, m, p = self.evaluate_metrics(st_data['Actual'], st_data['Predicted'])
                scores[f"RMSE_{station}"] = r
                scores[f"PVR_{station}"] = p

        # Calculate Temporal (Seasonal) Metrics
        for season in ['Summer', 'Winter']:
            sz_data = results_df[results_df['Season'] == season]
            if not sz_data.empty:
                r, m, p = self.evaluate_metrics(sz_data['Actual'], sz_data['Predicted'])
                scores[f"RMSE_{season}"] = r
                scores[f"PVR_{season}"] = p

        logger.info(f"Master Evaluation Complete: RMSE={overall_rmse:.4f}, PVR={overall_pvr:.2f}%")

        # 5. Generate Graphs
        self.generate_thesis_plots(results_df, scores)

        # 6. Log to MLflow
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            mlflow.log_metrics(scores)
            mlflow.log_metric("Learned_Ch", model.C_h.item())
            mlflow.log_metric("Learned_Csw", model.C_sw.item())

            # Save numeric metrics
            save_json(path=Path(self.config.metric_file_name), data=scores)

            # Log the visual graphs to MLflow
            mlflow.log_artifact(str(Path(self.config.root_dir) / "spatial_rmse_plot.png"))
            mlflow.log_artifact(str(Path(self.config.root_dir) / "actual_vs_predicted.png"))
            if Path("artifacts/model_trainer/training_loss_curve.png").exists():
                mlflow.log_artifact("artifacts/model_trainer/training_loss_curve.png")

            if tracking_url_type_store != "file":
                mlflow.pytorch.log_model(model, "model", registered_model_name="Greenland_LSTM_PINN")
            else:
                mlflow.pytorch.log_model(model, "model")

In [12]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
    
except Exception as e:
    logger.error(e)
    raise e

[2026-04-17 17:00:38,471: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-17 17:00:38,473: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-17 17:00:38,475: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-04-17 17:00:38,476: INFO: common: created directory at: artifacts]
[2026-04-17 17:00:38,478: INFO: common: created directory at: artifacts/model_evaluation]
[2026-04-17 17:00:38,478: INFO: 4265330102: Starting Evaluation and MLflow logging...]
[2026-04-17 17:00:38,679: INFO: 4265330102: Evaluation Metrics - RMSE: 5.4215, MAE: 4.1020, PVR: 0.00%]


2026/04/17 17:00:47 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/17 17:00:47 INFO mlflow.store.db.utils: Updating database tables


[2026-04-17 17:01:04,836: INFO: common: json file saved at: artifacts\model_evaluation\metrics.json]


2026/04/17 17:01:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/17 17:01:05 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/04/17 17:01:42 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'Greenland_LSTM_PINN'.
Created version '1' of model 'Greenland_LSTM_PINN'.
